# 3D Gaussian Splatting (3DGS) Baseline — Mip-NeRF 360 Multi-Resolution Sweep & Upload Pipeline

> **Kernel**: `thesis_env` (Set via Kernel → Change Kernel after running `bosch_setup_thesis.ipynb` once to register it)

Runs the full 3D Gaussian Splatting baseline training sweep on Mip-NeRF 360 scenes inside the BOSCH server environment across multiple selected image resolutions (`images`, `images_2`, `images_4`, `images_8`), automatically archiving, uploading results to Hugging Face, and purging local output files to save disk space.

**What this notebook does:**
1. **Proxy & Env Check**: Sets up environment variables for the BOSCH server.
2. **Target Resolutions Selection**: Choose any combination of resolutions (`["images", "images_2", "images_4", "images_8"]`).
3. **Resolution & Layout Validation**: Verifies selected resolution folders exist for all 9 scenes (`bicycle`, `flowers`, `garden`, `stump`, `treehill`, `room`, `counter`, `kitchen`, `bonsai`).
4. **Sweep Execution**: Invokes `run_mip360.sh` in `gaussian-splatting` for each selected resolution with `DATA_ROOT="/home/ghp4hc/datasets/datasets/mipneft360"`.
5. **Results Summary**: Formats and prints quantitative metrics (`results.json`) in a neat table for each resolution.
6. **Archiving & HF Upload**: Zips and uploads each resolution's output (`images.zip`, `images2.zip`, `images4.zip`, `images8.zip`) directly to `DiBiay/3dgs-mipnerf360-result`.
7. **Auto Cleanup**: Purges the local zip archive and output directory after a successful upload to save BOSCH server disk space.

## c00 — Proxy Settings
Sets the BOSCH proxy for external connectivity.

In [ ]:
# ── Proxy (required for HF / huggingface cache / diagnostic endpoints) ────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set to: {PROXY}')

## c01 — Config, Target Resolutions & Kernel Check
Defines paths, target resolution list (`["images", "images_2", "images_4", "images_8"]`), and double-checks if the correct virtual environment kernel is loaded.

In [ ]:
# ── Configurations & environment variables check ─────────────────────────────
import os
import sys

HOME = os.path.expanduser('~')

# Only running images_2 resolution
TARGET_RESOLUTIONS = ["images_2"]

# Robustly resolve 3DGS repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/gaussian-splatting'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/gaussian-splatting'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'gaussian-splatting')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'gaussian-splatting')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'gaussian-splatting')

ENV_NAME = 'thesis_env'

print(f'Active Python        : {sys.executable}')
print(f'Active Kernel name   : {ENV_NAME}')
print(f'Repository Root      : {REPO_ROOT}')
print(f'Target Resolutions   : {TARGET_RESOLUTIONS}')

assert REPO_ROOT in sys.executable or ENV_NAME in sys.executable or '.conda' in sys.executable, \
    f"WARNING: You are not running on the '{ENV_NAME}' kernel! Please select Kernel -> Change Kernel -> Python ({ENV_NAME})"

## c02 — Imports & GPU Validation
Verifies hardware detection and compiled custom modules availability.

In [ ]:
# ── Verification of PyTorch & custom submodules ────────────────────────────────
import torch
print('PyTorch version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device name :', torch.cuda.get_device_name(0))
    print('Compute Cap.    :', torch.cuda.get_device_capability(0))

import diff_gaussian_rasterization
import simple_knn
import fused_ssim
print('rasterizer      : OK')
print('simple-knn      : OK')
print('fused-ssim      : OK')

try:
    import huggingface_hub
    print('huggingface_hub : OK')
except ImportError:
    print('huggingface_hub : MISSING (will auto-install during the upload step)')

## c03 — Verify Dataset Path
Checks that the Mip-NeRF 360 source dataset is available on the server.

In [ ]:
# ── Verify Dataset Directory ──────────────────────────────────────────────────
import os
import sys

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
assert os.path.exists(src_root), f"Dataset path not found at {src_root}! Check that the datasets are downloaded."
print(f"✅ Found dataset source root: {src_root}")

print(f"\n📂 Source datasets directory content:")
print(os.listdir(src_root))

## c04 — Verify Target Resolutions Layout
Validates all 9 scenes for each resolution in `TARGET_RESOLUTIONS`.

In [ ]:
# ── Verify Mip-NeRF 360 Dataset Layout for all Target Resolutions ────────────
import os

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]

for resolution in TARGET_RESOLUTIONS:
    print(f"\n🔍 Verifying resolution '{resolution}' under {src_root} ...")
    missing = []
    for scene in MIP360_SCENES:
        v2_path = os.path.join(src_root, "360_v2", scene)
        extra_path = os.path.join(src_root, "360_extra_scenes", scene)
        
        if os.path.isdir(v2_path):
            scene_dir = v2_path
        elif os.path.isdir(extra_path):
            scene_dir = extra_path
        else:
            scene_dir = None
            
        if scene_dir is None:
            status = "MISSING (scene folder not found)"
            missing.append(scene)
        else:
            images_dir = os.path.join(scene_dir, resolution)
            if not os.path.isdir(images_dir):
                status = f"MISSING ({resolution} not found; has: {sorted(os.listdir(scene_dir))[:6]})"
                missing.append(scene)
            else:
                n_imgs = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                status = f"OK ({n_imgs} images)"
                
        print(f"  {scene:<12s} {status}")

    if missing:
        print(f"⚠️  {len(missing)}/{len(MIP360_SCENES)} scene(s) missing {resolution}: {missing}")
    else:
        print(f"✅ All {len(MIP360_SCENES)} scenes verified for '{resolution}'.")

## c05 — Run 3DGS Sweeps, Upload & Auto-Cleanup
Iterates through each resolution in `TARGET_RESOLUTIONS`, runs `run_mip360.sh`, summarizes metrics, zips output, uploads to Hugging Face, and purges local zip & output folder to save disk space.

In [ ]:
    # 3. Archive results into zip
    zip_out = os.path.join(os.path.dirname(REPO_ROOT), f"3dgs_output_mip360_{resolution}")
    archive_file = zip_out + ".zip"
    if os.path.isdir(out_root):
        # Strip point_cloud folders (large, unused after rendering) before archiving
        for scene in MIP360_SCENES:
            pc_dir = os.path.join(out_root, scene, "point_cloud")
            if os.path.isdir(pc_dir):
                shutil.rmtree(pc_dir, ignore_errors=True)

        print(f"\n📦 Archiving '{out_root}' -> '{archive_file}' ...")
        if os.path.exists(archive_file):
            os.remove(archive_file)
        shutil.make_archive(zip_out, "zip", out_root)
        print("Archived size:", round(os.path.getsize(archive_file) / 1e6, 1), "MB")
        
        # Prompt for HF token if not set
        token_to_use = HF_TOKEN
        if not token_to_use:
            token_to_use = input("Enter your Hugging Face Access Token (WRITE permission required): ").strip()
            
        # 4. Upload to Hugging Face
        if token_to_use:
            print(f"📤 Uploading '{archive_file}' as '{target_hf_filename}' to Hugging Face repo '{HF_REPO}'...")
            try:
                api = HfApi(proxies=PROXIES_DICT)
                try:
                    api.repo_info(repo_id=HF_REPO, repo_type="dataset", token=token_to_use)
                except Exception as repo_err:
                    if "404" in str(repo_err) or "Repository Not Found" in str(repo_err):
                        print(f"➕ Creating dataset repository '{HF_REPO}'...")
                        api.create_repo(repo_id=HF_REPO, repo_type="dataset", token=token_to_use, private=True)
                    else:
                        print(f"ℹ️ Repo info status note: {repo_err}")
                    
                api.upload_file(
                    path_or_fileobj=archive_file,
                    path_in_repo=target_hf_filename,
                    repo_id=HF_REPO,
                    repo_type="dataset",
                    token=token_to_use,
                )
                print(f"🎉 [SUCCESS] Uploaded '{target_hf_filename}' to '{HF_REPO}'!")
                
                # 5. Purge local zip and output directory to prevent OOM & save disk space
                print(f"🗑️ Cleaning up local zip file '{archive_file}' and output directory '{out_root}'...")
                if os.path.exists(archive_file):
                    os.remove(archive_file)
                if os.path.isdir(out_root):
                    shutil.rmtree(out_root, ignore_errors=True)
                print(f"✅ Local disk space freed for resolution '{resolution}'!")
            except Exception as e:
                print(f"❌ [ERROR] HF upload failed for {resolution}: {e}")
                print(f"⚠️  Retaining local zip '{archive_file}' and output folder '{out_root}' for inspection.")
        else:
            print(f"⚠️  Skipping HF upload (HF_TOKEN not set).")
    else:
        print(f"❌ [ERROR] Output directory '{out_root}' not found. Skipping zip/upload for {resolution}.")